# MLflow — First Contact

MLflow is a machine learning lifecycle platform that helps you manage experiments, track results, and deploy models. It has four core components: Tracking (log experiments), Projects (reproducible runs), Models (standard packaging), and Registry (model lifecycle management).

For a Data Engineer, ML is just another pipeline. You build features, feed models, track experiments, and serve predictions. MLflow sits in the middle of this flow as the system of record.

Citi framing: "Can we detect which endpoints are about to fail?" That becomes an anomaly detection problem. MLflow tracks experiments, logs models, and serves them.

```
[Postgres metrics] → [Feature Engineering] → [MLflow Experiment] → [Model Registry] → [Serving]
```

In [1]:
%pip install mlflow scikit-learn psycopg2-binary pandas numpy

import mlflow
import mlflow.sklearn
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import psycopg2
import pandas as pd
import numpy as np
import json

Note: you may need to restart the kernel to use updated packages.


## Verify MLflow tracking server is running

In [2]:
import requests

r = requests.get("http://localhost:5000/health")
print("MLflow Health:", r.status_code)

mlflow.set_tracking_uri("http://localhost:5000")
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow version:", mlflow.__version__)

MLflow Health: 200
MLflow tracking URI: http://localhost:5000
MLflow version: 3.10.1


## Feature Engineering from Telemetry Metrics

In [3]:

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="de_telemetry",
    user="de_admin",
    password="DeAdmin2026!"
)

metrics_df = pd.read_sql("SELECT * FROM metrics", conn)

pivot = pd.pivot_table(
    metrics_df,
    index="endpoint_id",
    columns="metric_name",
    values="value",
    aggfunc=["mean","std","max","min"]
)

pivot.columns = [f"{col[1]}_{col[0]}" for col in pivot.columns]

alerts_df = pd.read_sql("SELECT endpoint_id, COUNT(*) as alert_count FROM alerts GROUP BY endpoint_id", conn)

features_df = pivot.reset_index().merge(alerts_df, on="endpoint_id", how="left")
features_df = features_df.fillna(0)

feature_cols = [c for c in features_df.columns if c != "endpoint_id"]

print(f"Feature matrix: {features_df.shape[0]} endpoints × {features_df.shape[1]} features")
features_df.head()


C:\Users\shareuser\AppData\Local\Temp\ipykernel_20176\1817599808.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  metrics_df = pd.read_sql("SELECT * FROM metrics", conn)


Feature matrix: 10000 endpoints × 22 features


C:\Users\shareuser\AppData\Local\Temp\ipykernel_20176\1817599808.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  alerts_df = pd.read_sql("SELECT endpoint_id, COUNT(*) as alert_count FROM alerts GROUP BY endpoint_id", conn)


,endpoint_id,cpu_percent_mean,disk_io_mean,memory_percent_mean,network_in_mean,network_out_mean,cpu_percent_std,disk_io_std,memory_percent_std,network_in_std,...,disk_io_max,memory_percent_max,network_in_max,network_out_max,cpu_percent_min,disk_io_min,memory_percent_min,network_in_min,network_out_min,alert_count
0,1,77.220000,2298.211667,44.376250,566.017000,602.080000,25.543158,1646.953355,23.626517,214.050200,...,4840.99,80.84,780.71,936.19,35.78,577.97,15.96,134.32,268.39,0.0
1,2,37.481429,2573.100000,56.195385,545.891818,597.542941,16.657380,1843.060254,27.565869,298.827942,...,4875.92,98.70,903.82,958.77,16.11,277.98,13.76,43.76,82.72,3.0
2,3,60.800000,2194.776667,56.630000,425.398889,462.276154,30.923917,1498.306693,28.107077,349.629340,...,4434.46,96.60,931.06,971.69,15.84,299.37,15.30,49.95,30.00,4.0
3,4,49.429231,3030.298750,61.383125,565.880000,431.593750,28.698568,1687.250855,28.460527,222.843393,...,4956.83,97.69,877.67,882.30,13.01,255.78,8.07,251.54,91.25,3.0
4,5,53.881667,3327.908333,54.496923,658.168333,551.429412,31.944373,1281.081999,30.393941,246.893290,...,4620.22,93.08,967.38,989.61,12.61,957.98,8.98,268.70,50.40,1.0


## Experiment 1 — Isolation Forest

In [4]:

mlflow.set_experiment("citi_telemetry_anomaly")

with mlflow.start_run(run_name="isolation_forest_v1") as run:
    params = {"contamination":0.05,"n_estimators":100,"random_state":42}
    mlflow.log_params(params)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features_df[feature_cols])

    model = IsolationForest(**params)
    preds = model.fit_predict(X_scaled)

    anomaly_binary = (preds == -1).astype(int)

    mlflow.log_metric("anomaly_count", int(anomaly_binary.sum()))
    mlflow.log_metric("anomaly_rate", float(anomaly_binary.mean()))
    mlflow.log_metric("n_features", features_df.shape[1])

    mlflow.sklearn.log_model(scaler, "scaler")
    mlflow.sklearn.log_model(model, "model")
    mlflow.log_dict({"features": feature_cols}, "features.json")

    run_id = run.info.run_id
    print("Run ID:", run_id)


2026/03/31 18:13:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/03/31 18:13:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/03/31 18:13:21 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


2026/03/31 18:13:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/03/31 18:13:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: b31facfc69994b1e80cc03bb3a927f0c
🏃 View run isolation_forest_v1 at: http://localhost:5000/#/experiments/1/runs/b31facfc69994b1e80cc03bb3a927f0c
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Second Run

In [5]:

with mlflow.start_run(run_name="isolation_forest_v2"):
    params = {"contamination":0.1,"n_estimators":200,"random_state":42}
    mlflow.log_params(params)

    scaler2 = StandardScaler()
    X_scaled = scaler2.fit_transform(features_df[feature_cols])

    model2 = IsolationForest(**params)
    preds = model2.fit_predict(X_scaled)

    anomaly_binary = (preds == -1).astype(int)

    mlflow.log_metric("anomaly_count", int(anomaly_binary.sum()))
    mlflow.log_metric("anomaly_rate", float(anomaly_binary.mean()))

    mlflow.sklearn.log_model(model2, "model")


2026/03/31 18:13:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/03/31 18:13:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run isolation_forest_v2 at: http://localhost:5000/#/experiments/1/runs/2f49215fa0784cc2bc17245fa30c05ee
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Compare Runs

In [6]:

client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name("citi_telemetry_anomaly")

runs = client.search_runs(exp.experiment_id)

for r in runs:
    print(r.data.params, r.data.metrics)

print("UI: http://localhost:5000")


{'contamination': '0.1', 'n_estimators': '200', 'random_state': '42'} {'anomaly_count': 1000.0, 'anomaly_rate': 0.1}
{'contamination': '0.05', 'n_estimators': '100', 'random_state': '42'} {'anomaly_count': 500.0, 'anomaly_rate': 0.05, 'n_features': 22.0}
{'contamination': '0.1', 'n_estimators': '200', 'random_state': '42'} {'anomaly_count': 1000.0, 'anomaly_rate': 0.1}
{'contamination': '0.05', 'n_estimators': '100', 'random_state': '42'} {'anomaly_count': 500.0, 'anomaly_rate': 0.05, 'n_features': 22.0}
UI: http://localhost:5000


## Register Model

In [7]:

best_run = runs[0]
model_uri = f"runs:/{best_run.info.run_id}/model"

result = mlflow.register_model(model_uri, "citi_endpoint_anomaly_detector")

client.transition_model_version_stage(
    name="citi_endpoint_anomaly_detector",
    version=result.version,
    stage="Staging"
)

print("Model registered")


Registered model 'citi_endpoint_anomaly_detector' already exists. Creating a new version of this model...
2026/03/31 18:13:33 WARNING mlflow.tracking._model_registry.fluent: Run with id 2f49215fa0784cc2bc17245fa30c05ee has no artifacts at artifact path 'model', registering model based on models:/m-1beeff36ee344d12a5ab30e461c3df26 instead


2026/03/31 18:13:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: citi_endpoint_anomaly_detector, version 2


Created version '2' of model 'citi_endpoint_anomaly_detector'.


Model registered


C:\Users\shareuser\AppData\Local\Temp\ipykernel_20176\3650148951.py:6: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


## Load + Score

In [8]:

loaded_model = mlflow.sklearn.load_model("models:/citi_endpoint_anomaly_detector/Staging")

sample = features_df.sample(10, random_state=1)
preds = loaded_model.predict(sample[feature_cols])

anomalies = sample[preds == -1]

print("Anomalies:", anomalies["endpoint_id"].tolist())


Anomalies: [9954, 3851, 4963, 3887, 5438, 8518, 2042, 1990, 1934, 9985]


C:\py_venv\proj_educate\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(


## MLflow UI
http://localhost:5000

## Summary
Feature engineering → MLflow runs → model registry → serving